# WGW flood extent: reproducible walkthrough

This notebook is an inspectable front end to `build_flood_extent.py`. It shows the source files and metadata, the configured transformations, and the QC results from a selected scenario. The production algorithm stays in the script so the command line and notebook use the same implementation.

**Scientific caveat:** AQ confirmed `Val_1` is WSE in meters but did not confirm its vertical datum. NAVD88 remains a provisional assumption in the config. Do not interpret depth/extents as validated until the datum is confirmed.


## 1. Set paths and choose a scenario

Run from the `wgw-r4-demo-shell` root, or edit `REPO` to its absolute path. Keep the Shapefile sidecars together. Terrain defaults to the delivered processed 1 m GeoTIFF; edit `DTM` if it moves.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO = Path.cwd()
PIPELINE = REPO / "scripts/wgw_flood_pipeline"
DATA = REPO / "data/Max Water_Elevation_shp_9-10-2026"
DTM = Path("/Users/anthonygould/Documents/flood_sim/data/processed/terrain/expanded_coastal/wgw_expanded_coastal_usgs_1m_dtm.tif")

SCENARIOS = {
    "sandy": ("sandy_water_level_meters_10_29_2200.shp", "SCN-SANDY", "Hurricane Sandy"),
    "100yr": ("naccs_100_year_storm_water_level_meters_07_15_0200.shp", "SCN-100YR", "100-year storm"),
    "100yr_2050slr": ("naccs_100_year_storm_2050_slr_water_level_meters_07_15_0200.shp", "SCN-100YR-2050SLR", "100-year storm + 2050 SLR"),
    "100yr_2070slr": ("naccs_100_year_storm_2070_slr_water_level_meters_07_15_0200.shp", "SCN-100YR-2070SLR", "100-year storm + 2070 SLR"),
}
SCENARIO_KEY = "sandy"
WSE_FILE, SCENARIO_ID, SCENARIO_NAME = SCENARIOS[SCENARIO_KEY]
WSE = DATA / WSE_FILE
OUTPUT = REPO / "output" / SCENARIO_KEY

assert WSE.exists(), f"WSE input not found: {WSE}"
assert DTM.exists(), f"Terrain not found: {DTM}"
print("WSE:", WSE)
print("DTM:", DTM)
print("Output:", OUTPUT)


## 2. Inspect source metadata

This checks the vector CRS, field names, row count, raster CRS, pixel size, and NoData without loading the full raster into memory. The currently supplied scenario files are expected to have `Val_1` and EPSG:32618; the DTM is expected to use EPSG:26918 and 1 m pixels.


In [ ]:
import pyogrio
import rasterio

info = pyogrio.read_info(WSE)
vector_metadata = {"crs": str(info["crs"]), "fields": info["fields"].tolist(), "features": int(info["features"]), "geometry_type": info["geometry_type"]}
with rasterio.open(DTM) as src:
    terrain_metadata = {"crs": str(src.crs), "resolution": src.res, "shape": (src.height, src.width), "nodata": src.nodata, "bounds": tuple(src.bounds)}
print(json.dumps({"wse": vector_metadata, "dtm": terrain_metadata}, indent=2))


## 3. Review processing configuration and method

The script reprojects WSE polygons to the projected processing CRS (EPSG:26918), aligns terrain to it, and processes the result in raster windows. The depth equation is `WSE - DTM`; wet status is `depth > threshold`. Vector cleanup and grid splitting use meter units before conversion to the GeoJSON output CRS.


In [ ]:
CONFIG = json.loads((PIPELINE / "flood_extent_config.json").read_text())
print(json.dumps(CONFIG, indent=2))


## 4. Run the selected scenario

A full scenario run processes a large polygon set and raster subset, so allow time and disk space. Outputs go under `output/<scenario>/`, not into the source data directory. No source files are modified.


In [ ]:
command = [
    sys.executable, str(PIPELINE / "build_flood_extent.py"),
    "--wse-shp", str(WSE), "--dtm", str(DTM),
    "--scenario-id", SCENARIO_ID, "--scenario-name", SCENARIO_NAME,
    "--output-dir", str(OUTPUT), "--config", str(PIPELINE / "flood_extent_config.json"),
]
print(" ".join(command))
subprocess.run(command, check=True)


## 5. Review QC and generated artifacts

QC summaries are processing evidence, not scientific validation. Review the datum warning, area change, holes, vertices, maximum depth, and output part count. Then inspect the extent against source WSE and terrain in a GIS and trial the split GeoJSON in Nextspace.


In [ ]:
qc_path = OUTPUT / f"{SCENARIO_ID.lower()}_qc.json"
qc = json.loads(qc_path.read_text())
print(json.dumps(qc, indent=2))
print("\nOutput files:")
for path in sorted(OUTPUT.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")


## 6. Repeat for the other scenario types

Change `SCENARIO_KEY` in Section 1 to `100yr`, `100yr_2050slr`, or `100yr_2070slr`, then rerun Sections 1–5. Each scenario writes to its own output directory.
